# 导出 ONNX 并发布到 Hugging Face

这个 notebook 只做三件事：导出 ONNX、创建 Model 仓库、上传 Static Space。
不需要 GPU，选 Colab CPU 内核即可。

使用前确认：
1. Drive 的 `MyDrive/alexnet_cifar100/best_model.pth` 还在
2. 准备好 Hugging Face 的 Write token
3. 不要运行 `dev.ipynb` 的训练 cell

In [1]:
import torch
print(torch.__version__)
print("cuda:", torch.cuda.is_available())

2.14.1+cpu
cuda: False


## 1. 挂载 Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

## 2. 拉取最新代码

In [3]:
import os

if not os.path.exists('/content/alexnet-cifar100'):
    !git clone https://github.com/Wait4theWind/alexnet-cifar100.git

%cd /content/alexnet-cifar100
!git pull

Cloning into 'alexnet-cifar100'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 90 (delta 19), reused 75 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 240.15 KiB | 5.72 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/alexnet-cifar100
Already up to date.


In [4]:
!pip install -q 'huggingface_hub>=0.25,<2.0'

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.


## 3. 导出 ONNX

In [ ]:
!python src/export_onnx.py --weights /content/drive/MyDrive/alexnet_cifar100/best_model.pth --out space_static/alexnet_cifar100.onnx
!ls -lh space_static/alexnet_cifar100.onnx
!cp space_static/alexnet_cifar100.onnx /content/drive/MyDrive/alexnet_cifar100/

## 4. 登录 Hugging Face

运行下面的 cell 后粘贴 token。token 在 https://huggingface.co/settings/tokens 创建，类型选 Write。

In [ ]:
from huggingface_hub import login
login()

## 5. 创建仓库并上传

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.create_repo("alexnet-cifar100", repo_type="model", exist_ok=True)
api.create_repo("alexnet-cifar100-demo", repo_type="space", space_sdk="static", exist_ok=True)

api.upload_file(
    path_or_fileobj="/content/drive/MyDrive/alexnet_cifar100/best_model.pth",
    path_in_repo="best_model.pth",
    repo_id="bbbegemot/alexnet-cifar100",
    repo_type="model",
)
api.upload_file("src/model.py", "model.py", repo_id="bbbegemot/alexnet-cifar100", repo_type="model")
api.upload_file("model_card/README.md", "README.md", repo_id="bbbegemot/alexnet-cifar100", repo_type="model")

api.upload_folder(
    folder_path="space_static",
    repo_id="bbbegemot/alexnet-cifar100-demo",
    repo_type="space",
)

print("上传完成")

## 6. 测试

- Model: https://huggingface.co/bbbegemot/alexnet-cifar100
- Space: https://huggingface.co/spaces/bbbegemot/alexnet-cifar100-demo

Space 里上传图片，等模型加载完成后点“开始预测”。